# Ethical justification and value tensions in AI preference data
### Final implementation — Team Ethical Architects (Project 14, PW26)

**Target journal:** Ethics and Information Technology · **Anchor paper:** Dahlgren Lindström et al. (2025), doi 10.1007/s10676-025-09837-2 · **Dataset:** Anthropic/hh-rlhf (MIT), helpful-base and harmless-base, pinned revision `d8d97fe9…`

Enable Internet and run all cells. The two **test** files download automatically and are verified by SHA-256 against the pinned revision; attached local copies (test or train) are used when their content hashes match. If no test file is available the notebook falls back to the train split and says so in the provenance table.

| RQ | Question | Output |
|---|---|---|
| **RQ1a** | How do chosen and rejected replies differ in refusal, partial assistance, redirection and safety information in conflict pairs? | Fig 1a · `rq1a_strategies.csv` |
| **RQ1b** | How adequately are refusals justified (none / generic / specific risk)? | Fig 1b · `rq1b_justification.csv` |
| **RQ2** | Are hedged replies rejected in favour of confident ones in the helpful subset? | Fig 2 · `rq2_language.csv` |
| **RQ3** | Who bears the harm, and are third parties' interests acknowledged? | Fig 3 · `rq3_harm_bearer.csv`, `rq3_stakeholders.csv` |
| **RQ4** | Does the ranking stay stable, reverse or become unresolved under frozen ethical criteria? | Fig 4 · `rq4_reranking.csv`, `rq4_sensitivity.csv` |

**Methodology:** 1 configure · 2 load and verify · 3 clean and log exclusions · 4 lexicon features · 5 pilot and main samples, blinded coding sheets · 6 RQ1 · 7 RQ2 · 8 RQ3 · 9 RQ4 · 10 human-coding validation hook · 11 package.

All automated measures are lexical proxies that make the pipeline reproducible. They do not establish factual accuracy or ethical adequacy; the paper's claims rest on the blinded, double-coded human sample (step 10).

## 1 Configure paths, seed and output areas
Reproducibility settings, pinned source hashes, and separate folders for research outputs, diagnostics and annotation templates. Sample sizes follow the proposal: 40-pair pilot, 500-pair main sample, 25% double-coded.

In [ ]:
# Step M1: Configure local Kaggle paths, reproducibility and separate output areas.
from pathlib import Path
import json, gzip, hashlib, re, zipfile, platform, sys, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score
try:
    from IPython.display import display
except ImportError:  # Allows a non-Jupyter diagnostic harness; Kaggle supplies IPython.
    def display(obj):
        print(obj)

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = (Path('/kaggle/working/ethical_architects_dev')
             if Path('/kaggle').exists() else Path.cwd() / 'hh_rlhf_outputs')
AUTO_DOWNLOAD = True  # Fetch missing test files; disable for offline-only use.
DATA_CACHE = WORK_ROOT / 'dataset_cache'
DOWNLOAD_TIMEOUT = 30
# Optional exact paths. Otherwise gzip/plain JSONL is identified by uncompressed SHA-256.
DATA_PATHS = {}  # e.g. {'helpful_test': '/kaggle/input/my-data/test.jsonl.gz'}
HUMAN_INPUT = Path('/kaggle/input/ethical-architects-annotations')
SEED, N_BOOT = 20261026, 2000
PILOT_N, MAIN_N, DOUBLE_N, FACT_N = 20, 250, 125, 30
CODE_VERSION = 'ea-final-2.0'
ALLOW_TRAIN_FALLBACK = True  # use train split if no verified test file is available (offline runs)
RUN_TOKEN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN = WORK_ROOT / ('run_' + RUN_TOKEN)
PUBLIC, PRIVATE = RUN / 'research_outputs', RUN / 'private'
DIAG, EXPL, TEMPLATES = RUN / 'diagnostics', RUN / 'exploratory', RUN / 'annotation_templates'
for folder in [PUBLIC, PRIVATE, DIAG, EXPL, TEMPLATES]:
    folder.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'pdf.fonttype': 42, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 120})
SUBSETS = ['helpful', 'harmless']
SOURCES = {
 'helpful_test': (2354, '8be3fc1a13b27901631696f2be6f184c799f1baeaf145f53ac5db24960adc37b'),
 'harmless_test': (2312, 'ebfaaed21162a4de120ae85366075b34425e2a6303fac59b99555002f7016f03'),
 'helpful_train': (43835, '518a5bf288456fc9f3b7c980c54116fba0c52f274d3f4d344675d83e4058f6f4'),
 'harmless_train': (42537, 'bf3e1ecd3db61bf21d190f69351d26670c67aababa0fd282fa008c17a7202217')}
def digest_bytes(b):
    return hashlib.sha256(b).hexdigest()
def token(*parts):
    return digest_bytes('|'.join(map(str, parts)).encode())
def stable_rng(label):
    return np.random.default_rng(int(token(SEED, label)[:16], 16))
def save_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
environment = {'code_version': CODE_VERSION, 'seed': SEED, 'bootstrap_replicates': N_BOOT,
 'python': sys.version, 'platform': platform.platform(),
 'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__}
save_json(environment, DIAG / 'environment.json')
print('Development run:', RUN)
print('No human ethical or factual labels will be inferred by rules.')

## 2 Load and verify sources
Discovers gzip or plain JSONL files, verifies the uncompressed SHA-256 against the pinned Hugging Face revision, downloads the test files when absent, and writes a provenance table. Train files are accepted only when their hashes match.

In [ ]:
# Step M2: Discover gzip or Kaggle-decompressed JSONL; verify uncompressed UTF-8 content.
import glob

CONTENT_HASHES = {
 'helpful_test': '41c3a32fe45a7075f7e32b11bd29383f9fe650287354a95568f61295a8e694da',
 'harmless_test': '14d765196c9f18d84f9bb3a78bac608c8f2915110ebcbd74ec95db7b7198b008',
 'helpful_train': '99e5153cea0df0e94a8683f5f24031585d74412becff3206fe523dfee7d11734',
 'harmless_train': 'f6e548081aebe1974613025a1b49928d3d95a6450b28f4247010866adcc3e2dc',
}
unknown = set(DATA_PATHS) - set(SOURCES)
if unknown:
    raise ValueError(f'Unknown DATA_PATHS identities: {sorted(unknown)}')

def read_jsonl(path):
    opener = gzip.open if str(path).lower().endswith('.gz') else open
    try:
        with opener(path, 'rt', encoding='utf-8', errors='strict', newline='') as handle:
            return list(handle)
    except (OSError, UnicodeError, EOFError) as exc:
        raise ValueError(f'Cannot read gzip/UTF-8 JSONL file {path}: {exc}') from exc

content_cache = {}
discovery_log = []
def inspect_content(path):
    path=Path(path)
    if path not in content_cache:
        lines=read_jsonl(path)
        content_cache[path]=(lines, digest_bytes(''.join(lines).encode('utf-8')))
    return content_cache[path]

def find_file(identity):
    subset, split = identity.split('_')
    patterns = [f'**/{subset}_base_{split}.jsonl/{split}.jsonl',
                f'**/{subset}_base_{split}.jsonl', f'**/{subset}_base_{split}.jsonl.gz',
                f'**/{subset}-base/{split}.jsonl', f'**/{subset}-base/{split}.jsonl.gz',
                '**/*.jsonl', '**/*.gz']
    seen=set()
    for pattern in patterns:
        for hit in sorted(glob.glob(str(INPUT_ROOT / pattern), recursive=True)):
            path=Path(hit)
            if not path.is_file() or path in seen: continue
            seen.add(path)
            try:
                lines, h=inspect_content(path)
            except ValueError as exc:
                discovery_log.append({'identity':identity,'path':str(path),'status':'unreadable','detail':str(exc)})
                continue
            if h==CONTENT_HASHES[identity]:
                discovery_log.append({'identity':identity,'path':str(path),'status':'selected','detail':'verified uncompressed SHA-256'})
                return path
            discovery_log.append({'identity':identity,'path':str(path),'status':'not_this_identity','detail':h})
    return None

# Pin the source revision so future repository updates do not change this analysis.
from urllib.request import Request, urlopen
from urllib.error import URLError
SOURCE_REVISION = 'd8d97fe9e96fcd19cc8fc31bd04ee393422abfbb'
DOWNLOAD_URLS = {
    identity: f'https://huggingface.co/datasets/Anthropic/hh-rlhf/resolve/{SOURCE_REVISION}/{identity.split("_")[0]}-base/test.jsonl.gz'
    for identity in ['helpful_test', 'harmless_test']
}

def cached_or_downloaded_test(identity):
    """Return a verified cached test file, downloading it only when necessary."""
    cache_path = DATA_CACHE / (identity + '.jsonl.gz')
    if cache_path.is_file():
        _, h = inspect_content(cache_path)
        if h != CONTENT_HASHES[identity]:
            raise ValueError(f'Cached file is damaged: {cache_path}. Delete that cached file and rerun.')
        print(f'{identity}: using verified cached data')
        discovery_log.append({'identity': identity, 'path': str(cache_path),
                              'status': 'cached', 'detail': 'verified content'})
        return cache_path
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'{identity} is missing. Attach the test files or set AUTO_DOWNLOAD=True.')
    DATA_CACHE.mkdir(parents=True, exist_ok=True)
    url = DOWNLOAD_URLS[identity]
    print(f'{identity}: downloading official test data')
    try:
        request = Request(url, headers={'User-Agent': 'HH-RLHF-reproducible-analysis/1.0'})
        with urlopen(request, timeout=DOWNLOAD_TIMEOUT) as response:
            blob = response.read(20_000_001)
        if len(blob) > 20_000_000:
            raise ValueError('Unexpectedly large download; no cache file was written.')
        # Verify compressed and uncompressed bytes before saving any cache entry.
        if digest_bytes(blob) != SOURCES[identity][1]:
            raise ValueError('Downloaded file hash does not match the pinned original.')
        content = gzip.decompress(blob)
        if digest_bytes(content) != CONTENT_HASHES[identity]:
            raise ValueError('Downloaded JSONL content hash does not match the pinned original.')
        content.decode('utf-8', errors='strict')
        temporary = cache_path.with_suffix('.gz.part')
        temporary.write_bytes(blob)
        temporary.replace(cache_path)
    except (OSError, ValueError, EOFError, URLError) as exc:
        raise RuntimeError(
            f'Could not download verified {identity}: {exc}. '
            'Enable Internet in Kaggle settings and retry, or download the two '
            'test files manually and attach them as notebook input. '
            f'Source: {url}'
        ) from exc
    discovery_log.append({'identity': identity, 'path': str(cache_path),
                          'status': 'downloaded', 'detail': url})
    return cache_path

source_paths = {}
raw_records, schema_issues, provenance = [], [], []
for identity in SOURCES:
    explicit=identity in DATA_PATHS
    path=Path(DATA_PATHS[identity]) if explicit else find_file(identity)
    if path is None:
        if identity.endswith('_test'):
            try:
                path = cached_or_downloaded_test(identity)
            except RuntimeError as exc:
                if ALLOW_TRAIN_FALLBACK:
                    print(f'{identity}: unavailable ({str(exc)[:80]}...); will fall back to train split if present'); continue
                raise
        else:
            continue  # Optional training files are never automatically downloaded.
    if not path.is_file():
        raise FileNotFoundError(f'{identity}: DATA_PATHS file does not exist: {path}')
    lines,content_hash=inspect_content(path)
    if content_hash != CONTENT_HASHES[identity]:
        raise ValueError(f'{identity}: uncompressed contents differ from verified original; check file identity.')
    if len(lines) != SOURCES[identity][0]:
        raise ValueError(f'{identity}: unexpected line count {len(lines)}')
    subset, split = identity.split('_')
    valid=0
    for number,line in enumerate(lines,1):
        reason=''
        try:
            row=json.loads(line)
            if not isinstance(row,dict) or set(row)!={'chosen','rejected'}:
                reason='invalid_schema'
            elif any(not isinstance(v,str) or not v.strip() for v in row.values()):
                reason='missing_nonstring_or_empty_transcript'
        except ValueError:
            reason='blank_line' if not line.strip() else 'malformed_json'
        if reason:
            schema_issues.append({'source':identity,'line':number,'reason':reason})
        else:
            raw_records.append({'source':identity,'source_line':number,'subset':subset,'split':split,**row})
            valid+=1
    source_paths[identity]=path
    provenance.append({'identity':identity,'path':str(path),'sha256_content':content_hash,
      'sha256_stored_bytes':digest_bytes(path.read_bytes()),'storage_format':'gzip' if path.suffix.lower()=='.gz' else 'plain_jsonl',
      'source_url':DOWNLOAD_URLS.get(identity,''),'source_revision':SOURCE_REVISION,
      'bytes_stored':path.stat().st_size,'lines':len(lines),'schema_valid':valid,'schema_invalid':len(lines)-valid})
    print(f'{identity}: loaded {valid:,} verified records')
assert raw_records, 'No records loaded; do not proceed.'
pd.DataFrame(provenance).to_csv(DIAG/'source_provenance.csv',index=False)
pd.DataFrame(schema_issues,columns=['source','line','reason']).to_csv(DIAG/'schema_issues.csv',index=False)
pd.DataFrame(discovery_log,columns=['identity','path','status','detail']).to_csv(DIAG/'source_discovery.csv',index=False)
content_cache.clear()
display(pd.DataFrame(provenance))
print('Total loaded records:',len(raw_records))

## 3 Clean pairs and log exclusions
Parses transcripts, checks that chosen and rejected share the same context, removes empty, identical and repeated pairs, and records every exclusion in a data-flow table.

In [ ]:
# Step M3: Preserve complete conversation context and log sequential structural exclusions.
TURN_RE = re.compile(r'(?:\A|\n\n)(Human|Assistant):[ \t]*')
def parse_transcript(text):
    matches = list(TURN_RE.finditer(text))
    if not matches or text[:matches[0].start()].strip():
        return None
    return [(m.group(1), text[m.end():matches[i+1].start() if i+1 < len(matches) else len(text)].strip())
            for i, m in enumerate(matches)]

def parse_pair(row):
    c, r = parse_transcript(row['chosen']), parse_transcript(row['rejected'])
    reason = ''
    if not c or not r or c[-1][0] != 'Assistant' or r[-1][0] != 'Assistant':
        reason = 'unparseable_or_no_final_assistant'
    elif c[:-1] != r[:-1]:
        reason = 'context_mismatch'
    elif not c[-1][1] or not r[-1][1]:
        reason = 'empty_final_reply'
    elif c[-1][1] == r[-1][1]:
        reason = 'identical_final_replies'
    elif (len(c) < 2 or len(c) % 2 != 0 or
          any(role != ('Human' if i % 2 == 0 else 'Assistant') for i,(role,_) in enumerate(c)) or
          any(not t for _, t in c[:-1])):
        reason = 'nonalternating_or_empty_context_turn'
    out = {k: row[k] for k in ['source','source_line','subset','split']}
    out['reason'] = reason
    if not reason:
        context = json.dumps(c[:-1], ensure_ascii=False, separators=(',', ':'))
        out.update(context_hash=token(context), context='\n\n'.join(f'{role}: {t}' for role,t in c[:-1]),
                   chosen_resp=c[-1][1], rejected_resp=r[-1][1])
        out['record_hash'] = token(context, c[-1][1], r[-1][1])
    return out

parsed_records = [parse_pair(row) for row in raw_records]
flow, exclusions, repeated, clean_parts = [], [], [], []
for source in sorted(source_paths):
    all_rows = [r for r in parsed_records if r['source'] == source]
    invalid_schema = sum(r['source'] == source for r in schema_issues)
    flow.append({'source': source, 'step': 'raw_lines', 'excluded': 0, 'retained': len(all_rows)+invalid_schema})
    flow.append({'source': source, 'step': 'schema', 'excluded': invalid_schema, 'retained': len(all_rows)})
    remaining = len(all_rows)
    reasons = ['unparseable_or_no_final_assistant','context_mismatch','empty_final_reply',
               'identical_final_replies','nonalternating_or_empty_context_turn']
    for reason in reasons:
        bad = [r for r in all_rows if r['reason'] == reason]
        exclusions.extend({k: r[k] for k in ['source','source_line','reason']} for r in bad)
        remaining -= len(bad)
        flow.append({'source':source,'step':reason,'excluded':len(bad),'retained':remaining})
    good = pd.DataFrame([r for r in all_rows if not r['reason']])
    if good.empty:
        raise ValueError(f'No structurally usable pairs in {source}.')
    for h,g in good.groupby('context_hash'):
        if len(g)>1:
            repeated.append({'source':source,'context_hash':h,'records':len(g),
                             'exact_pair_surplus':int(g.record_hash.duplicated().sum())})
    # Sorting makes retained representatives deterministic, independent of filesystem discovery order.
    good = good.sort_values(['record_hash','source_line'])
    dup = good[good.context_hash.duplicated()]
    exclusions.extend({'source':source,'source_line':int(r.source_line),'reason':'repeated_context'}
                      for r in dup.itertuples())
    good = good.drop_duplicates('context_hash').copy()
    flow.append({'source':source,'step':'repeated_context','excluded':len(dup),'retained':len(good)})
    clean_parts.append(good)
clean = pd.concat(clean_parts, ignore_index=True)
clean['pair_id'] = clean.apply(lambda r: 'p_' + token(r.source, r.record_hash)[:20], axis=1)
assert clean.pair_id.is_unique
overlap = []
for i, a in enumerate(sorted(source_paths)):
    for b in sorted(source_paths)[i+1:]:
        overlap.append({'source_a':a,'source_b':b,'shared_clean_contexts':len(
          set(clean.loc[clean.source==a,'context_hash']) & set(clean.loc[clean.source==b,'context_hash']))})
pd.DataFrame(overlap).to_csv(DIAG / 'context_overlap.csv', index=False)
pd.DataFrame(flow).to_csv(DIAG / 'cleaning_flow.csv', index=False)
pd.DataFrame(exclusions).to_csv(DIAG / 'exclusions.csv', index=False)
pd.DataFrame(repeated, columns=['source','context_hash','records','exact_pair_surplus']).to_csv(DIAG / 'repeated_contexts.csv', index=False)
ANALYSIS_SPLIT = 'test' if (clean.split == 'test').any() else 'train'
if ANALYSIS_SPLIT == 'train':
    print('WARNING: no verified test file loaded; analysing the TRAIN split instead. The proposal specifies the test split.')
pool = clean[clean.split == ANALYSIS_SPLIT].sort_values(['subset','record_hash']).copy()
save_json({'analysis_split': ANALYSIS_SPLIT, 'pairs': int(len(pool))}, DIAG / 'analysis_split.json')
# Global test-context deduplication prevents pilot/main cross-subset context leakage.
cross = pool[pool.context_hash.duplicated()].copy()
cross[['pair_id','subset','context_hash']].to_csv(DIAG / 'cross_subset_test_exclusions.csv', index=False)
pool = pool.drop_duplicates('context_hash').reset_index(drop=True)
display(pd.DataFrame(flow)); display(pd.DataFrame(overlap))
print('Conservative structural screen: quoted role markers may cause false exclusions; inspect logs before freezing.')

## 4 Lexicon features (step 6 of the methodology)
Transparent regular-expression lexicons, listed in full: resolution features (multi-label), justification level of refusals, hedging and assertion markers, checkable-claim density, harm bearer of the request, and acknowledgment of a third party's interests.

In [ ]:
# Extract features using the original lexicon definitions.
df = pool.copy()
df['last_user'] = df.context.map(lambda x: x.rsplit('Human:',1)[-1].strip())
L = {
 "refuse":   r"\b(i can(?:'|no)?t|i cannot|i won(?:'|no)?t|i will not|i(?:'m| am) not (?:able|going|comfortable|willing)|"
             r"i(?:'m| am) sorry,? but|i don(?:'|no)?t think (?:i|that) (?:should|can)|not (?:something i|able to help)|"
             r"i (?:would|must) (?:not|refuse)|that(?:'s| is) (?:not appropriate|not okay|illegal|unethical|harmful|dangerous)|"
             r"i(?:'d| would) (?:rather not|prefer not)|can'?t help (?:you )?with (?:that|this))\b",
 "redirect": r"\b(instead|alternatively|i(?:'d| would) (?:recommend|suggest)|you (?:could|might|should) (?:try|consider|talk|speak|contact|call|see|reach)|"
             r"professional|therapist|counsel(?:l)?or|doctor|hotline|lawyer|police|911|emergency|support line|mental health)\b",
 "help":     r"\b(here(?:'s| is| are)|you can|first,?|step|for example|one way|the best way|try (?:to|this)|"
             r"to do (?:this|that)|you(?:'ll| will) need|what you (?:want|need) to do|you should)\b",
 "safety":   r"\b(be careful|safely|safety|safe way|risk|dangerous|harm(?:ful)?|caution|protect|consult|supervis|"
             r"legal(?:ly)?|consequence|warning|precaution)\b",
 "risk_named": r"\b(illegal|against the law|crime|hurt|harm|injur|danger|unsafe|privacy|consent|discriminat|offensive|"
               r"abus|violence|violent|dead|death|kill|poison|weapon|steal|fraud|hate|racis|sexual|minor|child)\b",
 "generic_reason": r"\b(because|since|as it|i(?:'m| am) (?:an ai|a language model|not able)|not appropriate|not something i|"
                   r"i(?:'m| am) not (?:comfortable|supposed))\b",
 "hedge":    r"\b(i think|i believe|i(?:'m| am) not (?:sure|certain)|not (?:entirely |completely |100% )?(?:sure|certain)|"
             r"might|maybe|perhaps|possibly|probably|it depends|i don(?:'|no)?t know|as far as i know|"
             r"i(?:'m| am) not an expert|could be|seems?|roughly|approximately|generally|typically|in general)\b",
 "assert":   r"\b(definitely|certainly|absolutely|always|never|clearly|obviously|undoubtedly|of course|"
             r"it is (?:well )?known|the fact is|in fact|without (?:a )?doubt|guaranteed|exactly|precisely|100%)\b",
 "claim":    r"(\b\d[\d,\.]*\b|\b(?:19|20)\d\d\b|\b[A-Z][a-z]{3,}\b)",
 "harm_user":    r"\b(myself|my (?:own )?(?:life|body|health)|kill me|suicid|self[- ]harm|overdose|starv|i want to die|hurt myself|purge)\b",
 "harm_third":   r"\b(my (?:wife|husband|girlfriend|boyfriend|ex|neighbou?r|boss|coworker|friend|sister|brother|mom|dad|kid|child|teacher|roommate)|"
                 r"someone|somebody|a (?:girl|boy|woman|man|person|kid|child)|him|her|them|their|people who|"
                 r"get back at|revenge|without (?:them|him|her) knowing|stalk|hurt (?:him|her|them|someone)|steal from|break into|beat up|kidnap)\b",
 "harm_society": r"\b(bomb|explosive|terror|mass (?:shooting|casualt)|election|voters?|government|public|society|"
                 r"pandemic|virus|bioweapon|chemical weapon|hack(?:ing)? (?:the|into)|infrastructure|counterfeit|launder|"
                 r"fake news|propaganda|racis|genocide|ethnic|immigrants?|minorit)\b",
 "ack_third":    r"\b(their (?:privacy|consent|feelings|rights|safety|wellbeing|well-being|life|perspective)|"
                 r"(?:his|her|their) (?:own )?(?:choice|decision|autonomy)|without (?:their|his|her) (?:consent|permission|knowledge)|"
                 r"the other person|other people|how (?:they|he|she) (?:feel|would feel)|respect (?:them|him|her|their)|"
                 r"(?:hurt|harm|affect) (?:them|him|her|others|other people|someone else)|not fair to|their side)\b",
}
L = {k: re.compile(v, re.I if k != "claim" else 0) for k, v in L.items()}

def feats(text):
    t = text.lower(); n = max(1, len(text.split()))
    ref, red, hlp, saf = (bool(L[k].search(t)) for k in ("refuse", "redirect", "help", "safety"))
    just = "n/a"
    if ref:
        just = "specific" if L["risk_named"].search(t) else ("generic" if L["generic_reason"].search(t) else "none")
    return {"refuse": ref, "partial": ref and hlp, "redirect": red, "safety_info": saf, "justification": just,
            "hedge": 100*len(L["hedge"].findall(t))/n, "assert": 100*len(L["assert"].findall(t))/n,
            "claims": 100*len(L["claim"].findall(text))/n, "ack_third": bool(L["ack_third"].search(t)), "words": n}

def harm_bearer(p):
    p = p.lower()
    if L["harm_society"].search(p): return "society"
    if L["harm_third"].search(p):   return "third party"
    if L["harm_user"].search(p):    return "requester"
    return "none detected"

for side in ("chosen", "rejected"):
    F = pd.DataFrame([feats(x) for x in df[f"{side}_resp"]], index=df.index).add_prefix(f"{side}_")
    df = df.join(F)
df["harm_bearer"] = df.last_user.map(harm_bearer)
df["conflict"]    = (df.subset == "harmless") | (df.harm_bearer != "none detected")
df["third_case"]  = df.harm_bearer == "third party"
print("conflict pairs:", int(df.conflict.sum()), "| third-party cases:", int(df.third_case.sum()))

## 5 Pilot and main samples, blinded coding sheets (step 4)
Draws the 40-pair pilot and the 500-pair main sample with the fixed seed, disjoint from each other. Coding sheets hide the chosen/rejected label and show the two replies as Response A/B in random order; the key is written to the private folder.

In [ ]:
def strat(frame, n, exclude=None, label=''):
    pool_ = frame if exclude is None else frame[~frame.pair_id.isin(exclude)]
    parts = []
    for sub, g in pool_.groupby('subset'):
        idx = stable_rng(label + sub).choice(g.index.to_numpy(), size=min(n, len(g)), replace=False)
        parts.append(g.loc[idx])
    return pd.concat(parts).reset_index(drop=True)

pilot = strat(df, PILOT_N, label='pilot')
main  = strat(df, MAIN_N, exclude=pilot.pair_id, label='main')
double = strat(main, DOUBLE_N, label='double')   # the 25% double-coding subset

ANNOT = ['A_refuse','A_partial','A_redirect','A_safety_info','A_justification(none/generic/specific)',
         'A_truth(supported/contradicted/unresolved/no_claim)','A_uncertainty(hedged/neutral/assertive)',
         'B_refuse','B_partial','B_redirect','B_safety_info','B_justification(none/generic/specific)',
         'B_truth(supported/contradicted/unresolved/no_claim)','B_uncertainty(hedged/neutral/assertive)',
         'conflict_present(y/n)','harm_bearer(none/requester/third_party/society)',
         'third_party_interests_acknowledged(A/B/both/neither)','ethical_rank(A/B/tie/insufficient_evidence)','coder_notes']

def blinded(frame, name):
    rng = stable_rng('blind' + name); flip = rng.random(len(frame)) < 0.5
    sheet = pd.DataFrame({'pair_id': frame.pair_id, 'subset': frame.subset, 'context': frame.context, 'last_user': frame.last_user,
                          'response_A': np.where(flip, frame.rejected_resp, frame.chosen_resp),
                          'response_B': np.where(flip, frame.chosen_resp, frame.rejected_resp)})
    for c in ANNOT: sheet[c] = ''
    sheet.to_csv(TEMPLATES / f'{name}_blinded.csv', index=False)
    pd.DataFrame({'pair_id': frame.pair_id, 'A_is': np.where(flip, 'rejected', 'chosen')}).to_csv(PRIVATE / f'{name}_KEY.csv', index=False)

blinded(pilot, 'pilot_40'); blinded(main, 'main_500')
double[['pair_id','subset']].to_csv(TEMPLATES / 'double_coding_subset_ids.csv', index=False)
main[['pair_id','subset']].to_csv(TEMPLATES / 'main_sample_ids.csv', index=False)
print('pilot:', pilot.subset.value_counts().to_dict(), '| main:', main.subset.value_counts().to_dict(), '| double-coded:', len(double))
print('Blinded sheets ->', TEMPLATES, '| keys ->', PRIVATE)

## 6 RQ1 — Resolution strategies (1a) and their justification (1b)
Paired comparison of chosen vs rejected replies in conflict pairs: bootstrap CIs on the paired difference, exact McNemar tests with Holm correction, and odds that the feature-bearing reply is the chosen one. Figure 1b reports the justification level of refusals.

In [ ]:
from scipy.stats import binomtest, chi2_contingency, wilcoxon
RESULTS = RUN / 'research_outputs'; manifest = []
NAVY, GOLD, TEAL, GREY = '#17365D', '#D79B27', '#00A6A6', '#AAB3BC'

def holm_adjust(p):
    p = np.asarray(p, float); order = np.argsort(p); out = np.empty(len(p)); last = 0.
    for j, i in enumerate(order):
        last = max(last, min(1., p[i] * (len(p) - j))); out[i] = last
    return out
def boot_ci(v, label):
    v = np.asarray(v, float); rng = stable_rng(label)
    if len(v) == 0: return np.nan, np.nan, np.nan
    b = [rng.choice(v, len(v)).mean() for _ in range(N_BOOT)]
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)
def odds_chosen(fc, fr):
    a, b = int((fc & ~fr).sum()), int((~fc & fr).sum())
    if a == 0 or b == 0: return np.nan, np.nan, np.nan, a, b
    o = a / b; se = np.sqrt(1/a + 1/b); return o, np.exp(np.log(o)-1.96*se), np.exp(np.log(o)+1.96*se), a, b
def save(fig, name):
    for ext in ['pdf', 'png']: fig.savefig(RESULTS / f'{name}.{ext}', bbox_inches='tight', dpi=180)
    manifest.append(name)

FEATS = [('refuse','Refusal'),('partial','Refusal + help'),('redirect','Redirection'),('safety_info','Safety information')]
conf = df[df.conflict].copy(); rows = []
for sub, g in conf.groupby('subset'):
    for f, lab in FEATS:
        c, r = g['chosen_'+f].astype(float), g['rejected_'+f].astype(float)
        m, lo, hi = boot_ci(c - r, 'rq1a'+sub+f)
        o, olo, ohi, a, b = odds_chosen(c == 1, r == 1)
        p = binomtest(a, a+b, .5).pvalue if a + b else np.nan
        rows.append(dict(subset=sub, feature=lab, n_pairs=len(g), chosen_pct=100*c.mean(), rejected_pct=100*r.mean(),
                         diff_pct=100*m, ci_low=100*lo, ci_high=100*hi, OR_feature_is_chosen=o, OR_low=olo, OR_high=ohi,
                         only_in_chosen=a, only_in_rejected=b, exact_mcnemar_p=p))
t1a = pd.DataFrame(rows); v = t1a.exact_mcnemar_p.notna(); t1a['holm_p'] = np.nan; t1a.loc[v, 'holm_p'] = holm_adjust(t1a.loc[v, 'exact_mcnemar_p'])
t1a.round(4).to_csv(RESULTS / 'rq1a_strategies.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, sub in zip(axes, SUBSETS):
    s = t1a[t1a.subset == sub]; x = np.arange(len(s)); w = .36
    ax.bar(x-w/2, s.chosen_pct, w, color=NAVY, label='Chosen'); ax.bar(x+w/2, s.rejected_pct, w, color=GOLD, label='Rejected')
    for xi, (_, rr) in zip(x, s.iterrows()):
        ax.errorbar(xi, max(rr.chosen_pct, rr.rejected_pct) + 0.4, yerr=[[0], [0]], fmt='none')
        ax.text(xi, max(rr.chosen_pct, rr.rejected_pct) + 0.6, f'OR {rr.OR_feature_is_chosen:.2f}' if not np.isnan(rr.OR_feature_is_chosen) else 'OR n/a', ha='center', fontsize=8, color='#333')
    ax.set_xticks(x); ax.set_xticklabels(s.feature, rotation=20, ha='right'); ax.set_title(f'{sub} conflict pairs (n={int(s.n_pairs.iloc[0]) if len(s) else 0:,})')
axes[0].set_ylabel('% of replies with feature'); axes[0].legend(frameon=False)
fig.suptitle('Figure 1a. Resolution features in chosen vs rejected replies (lexicon pass)'); fig.tight_layout(); save(fig, 'fig1a_rq1a_strategies'); plt.show(); plt.close(fig)

JUST = ['none', 'generic', 'specific']
long = pd.concat([pd.DataFrame({'side': 'chosen', 'just': conf.loc[conf.chosen_refuse, 'chosen_justification']}),
                  pd.DataFrame({'side': 'rejected', 'just': conf.loc[conf.rejected_refuse, 'rejected_justification']})], ignore_index=True)
ct1b = pd.crosstab(long.side, long.just).reindex(index=['chosen','rejected'], columns=JUST, fill_value=0)
pct1b = 100 * ct1b.div(ct1b.sum(axis=1).replace(0, np.nan), axis=0)
try: chi2v, p1b, _, _ = chi2_contingency(ct1b)
except ValueError: chi2v, p1b = np.nan, np.nan
both = conf[conf.chosen_refuse & conf.rejected_refuse]
o, lo, hi, a, b = odds_chosen(both.chosen_justification == 'specific', both.rejected_justification == 'specific')
t1b = pd.concat({'count': ct1b, 'row_percent': pct1b.round(1)}, axis=1); t1b.to_csv(RESULTS / 'rq1b_justification.csv')
save_json({'chi2': chi2v, 'p': p1b, 'both_refuse_pairs': int(len(both)), 'OR_specific_reason_is_chosen': o, 'OR_ci': [lo, hi]}, RESULTS / 'rq1b_stats.json')
fig, ax = plt.subplots(figsize=(7, 3.2)); left = np.zeros(2); cols = {'none': GREY, 'generic': GOLD, 'specific': NAVY}
for j in JUST:
    vals = pct1b[j].fillna(0).values; ax.barh(['chosen', 'rejected'], vals, left=left, color=cols[j], label=j); left += vals
ax.set_xlabel('% of refusals'); ax.legend(frameon=False, ncol=3, loc='lower center', bbox_to_anchor=(.5, 1.0))
ax.set_title(f'Figure 1b. Justification level of refusals (chosen n={int(ct1b.loc["chosen"].sum())}, rejected n={int(ct1b.loc["rejected"].sum())})', pad=26)
fig.tight_layout(); save(fig, 'fig1b_rq1b_justification'); plt.show(); plt.close(fig)
display(t1a.round(3)); display(t1b)

## 7 RQ2 — Expressed uncertainty and claim density (helpful subset)
Hedging, confident-assertion and checkable-claim markers per 100 words, chosen vs rejected, with bootstrap CIs on the paired difference, paired Wilcoxon tests, and odds that the more-hedged / more-assertive / claim-denser reply is chosen. Factual support (supported / contradicted / unresolved) comes from the human fact-check in step 10.

In [ ]:
hp = df[df.subset == 'helpful'].copy(); rows = []
for f, lab in [('hedge','Hedges / 100 words'),('assert','Assertions / 100 words'),('claims','Checkable claims / 100 words')]:
    c, r = hp['chosen_'+f], hp['rejected_'+f]
    m, lo, hi = boot_ci(c - r, 'rq2'+f)
    try: wp = wilcoxon(c, r).pvalue
    except ValueError: wp = np.nan
    o, olo, ohi, a, b = odds_chosen(c > r, r > c)
    rows.append(dict(feature=lab, n_pairs=len(hp), chosen=c.mean(), rejected=r.mean(), diff=m, ci_low=lo, ci_high=hi,
                     wilcoxon_p=wp, OR_moreX_is_chosen=o, OR_low=olo, OR_high=ohi, a=a, b=b))
o0, l0, h0, a0, b0 = odds_chosen(hp.chosen_hedge == 0, hp.rejected_hedge == 0)
rows.append(dict(feature='No hedging at all', n_pairs=len(hp), OR_moreX_is_chosen=o0, OR_low=l0, OR_high=h0, a=a0, b=b0))
t2 = pd.DataFrame(rows); t2.round(4).to_csv(RESULTS / 'rq2_language.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.3, 1]})
s = t2.iloc[:3]; x = np.arange(3); w = .36
ax1.bar(x-w/2, s.chosen, w, color=NAVY, label='Chosen'); ax1.bar(x+w/2, s.rejected, w, color=GOLD, label='Rejected')
ax1.set_xticks(x); ax1.set_xticklabels(['hedging', 'assertion', 'claim density']); ax1.set_ylabel('markers per 100 words')
ax1.set_title(f'(a) Helpful subset, n={len(hp):,} pairs'); ax1.legend(frameon=False)
for i, (_, rr) in enumerate(t2.iterrows()):
    if np.isnan(rr.OR_moreX_is_chosen): continue
    ax2.errorbar(rr.OR_moreX_is_chosen, i, xerr=[[rr.OR_moreX_is_chosen-rr.OR_low], [rr.OR_high-rr.OR_moreX_is_chosen]], fmt='o', color=NAVY, capsize=3)
ax2.set_yticks(range(len(t2))); ax2.set_yticklabels(['more hedged', 'more assertive', 'more claims', 'no hedging']); ax2.axvline(1, color='grey', ls='--', lw=1); ax2.set_xscale('log')
ax2.set_xlabel('odds this reply is the chosen one'); ax2.set_title('(b) Preference odds, discordant pairs')
fig.suptitle('Figure 2. Expressed uncertainty vs confidence in chosen replies (lexicon pass)'); fig.tight_layout(); save(fig, 'fig2_rq2_language'); plt.show(); plt.close(fig)
display(t2.round(3))

## 8 RQ3 — Harm bearer and acknowledgment of third parties
(a) Dominant resolution of the chosen reply by who would be harmed (requester / third party / society), with χ² and Cramér's V. (b) In third-party cases, share of chosen vs rejected replies acknowledging the third party's interests, with exact McNemar test and odds ratio.

In [ ]:
def dominant(r, side):
    if r[f'{side}_refuse'] and r[f'{side}_partial']: return 'partial'
    if r[f'{side}_refuse'] and r[f'{side}_redirect']: return 'redirect'
    if r[f'{side}_refuse']: return 'refuse'
    if r[f'{side}_redirect']: return 'redirect'
    return 'comply'
ORDER = ['refuse', 'redirect', 'partial', 'comply']; BEARERS = ['requester', 'third party', 'society']
df['chosen_dom'] = df.apply(lambda r: dominant(r, 'chosen'), axis=1); df['rejected_dom'] = df.apply(lambda r: dominant(r, 'rejected'), axis=1)
hm = df[df.conflict & df.harm_bearer.isin(BEARERS)]
ct3 = pd.crosstab(hm.harm_bearer, hm.chosen_dom).reindex(index=BEARERS, columns=ORDER, fill_value=0)
pct3 = 100 * ct3.div(ct3.sum(axis=1).replace(0, np.nan), axis=0)
try:
    chi3, p3, dof3, _ = chi2_contingency(ct3); v3 = np.sqrt(chi3 / (ct3.values.sum() * (min(ct3.shape) - 1)))
except ValueError: chi3 = p3 = dof3 = v3 = np.nan
pd.concat({'count': ct3, 'row_percent': pct3.round(1)}, axis=1).to_csv(RESULTS / 'rq3_harm_bearer.csv')

rows = []
for sub, g in df[df.third_case].groupby('subset'):
    c, r = g.chosen_ack_third.astype(float), g.rejected_ack_third.astype(float)
    m, lo, hi = boot_ci(c - r, 'rq3'+sub); o, olo, ohi, a, b = odds_chosen(c == 1, r == 1)
    rows.append(dict(subset=sub, n_pairs=len(g), chosen_pct=100*c.mean(), rejected_pct=100*r.mean(), diff_pct=100*m, ci_low=100*lo, ci_high=100*hi,
                     OR_ack_is_chosen=o, OR_low=olo, OR_high=ohi, only_in_chosen=a, only_in_rejected=b, exact_mcnemar_p=binomtest(a, a+b, .5).pvalue if a+b else np.nan))
t3 = pd.DataFrame(rows); t3.round(4).to_csv(RESULTS / 'rq3_stakeholders.csv', index=False)
save_json({'chi2': chi3, 'dof': dof3, 'p': p3, 'cramers_v': v3}, RESULTS / 'rq3_stats.json')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.4, 1]})
im = ax1.imshow(pct3.fillna(0).values, cmap='Blues', aspect='auto', vmin=0)
ax1.set_xticks(range(4)); ax1.set_xticklabels(ORDER); ax1.set_yticks(range(3)); ax1.set_yticklabels([f'{b_} (n={int(ct3.loc[b_].sum())})' for b_ in BEARERS])
for i in range(3):
    for j in range(4):
        val = pct3.fillna(0).values[i, j]; ax1.text(j, i, f'{val:.0f}%', ha='center', va='center', color='white' if val > 55 else 'black', fontsize=9)
ax1.set_xlabel('dominant resolution of the CHOSEN reply'); ax1.set_title(f'(a) By who would be harmed  (χ²={chi3:.1f}, p={p3:.2g}, V={v3:.2f})')
x = np.arange(len(t3)); w = .36
ax2.bar(x-w/2, t3.chosen_pct, w, color=NAVY, label='Chosen'); ax2.bar(x+w/2, t3.rejected_pct, w, color=GOLD, label='Rejected')
ax2.set_xticks(x); ax2.set_xticklabels([f'{s} (n={n})' for s, n in zip(t3.subset, t3.n_pairs)]); ax2.set_ylabel('% acknowledging third party'); ax2.legend(frameon=False)
ax2.set_title('(b) Third-party cases: acknowledgment')
fig.suptitle('Figure 3. Whose harm counts, and whether absent parties are acknowledged (lexicon pass)'); fig.tight_layout(); save(fig, 'fig3_rq3_harm_bearer'); plt.show(); plt.close(fig)
display(pd.concat({'count': ct3, 'row_percent': pct3.round(1)}, axis=1)); display(t3.round(3))

## 9 RQ4 — Ethical re-ranking under frozen criteria
Three criteria scored per reply from the lexicon codes and combined by a lexicographic rule (harm prevention > respect for affected parties > factual support): **stable** (criteria prefer the chosen reply), **reversed**, **unresolved** (tie on every applicable criterion), **insufficient evidence** (no criterion applies). Sensitivity across all six orderings, plus the earlier two-rule check. In the paper the same procedure is run on the human codes.

In [ ]:
import itertools
CRITERION_ORDER = ['harm_prevention', 'respect_affected', 'factual_support']
def crit(r, side):
    s = {}
    if r.conflict:   s['harm_prevention'] = 1 if (r[f'{side}_refuse'] or r[f'{side}_redirect'] or r[f'{side}_safety_info']) else -1
    if r.third_case: s['respect_affected'] = 1 if r[f'{side}_ack_third'] else 0
    if r.subset == 'helpful': s['factual_support'] = r[f'{side}_hedge'] - r[f'{side}_assert']
    return s
def rerank(r, order):
    sc, sr = crit(r, 'chosen'), crit(r, 'rejected')
    if not sc: return 'insufficient evidence'
    for c in order:
        if c in sc and sc[c] != sr[c]: return 'stable' if sc[c] > sr[c] else 'reversed'
    return 'unresolved'
OUTC = ['stable', 'reversed', 'unresolved', 'insufficient evidence']
df['rq4_outcome'] = df.apply(lambda r: rerank(r, CRITERION_ORDER), axis=1)
rows = []
for name, g in list(df.groupby('subset')) + [('all', df)] + [(f'bearer:{b_}', df[df.harm_bearer == b_]) for b_ in BEARERS]:
    for o_ in OUTC:
        m, lo, hi = boot_ci((g.rq4_outcome == o_).astype(float), 'rq4'+name+o_)
        rows.append(dict(group=name, outcome=o_, n=len(g), share_pct=100*m, ci_low=100*lo, ci_high=100*hi))
t4 = pd.DataFrame(rows); t4.round(3).to_csv(RESULTS / 'rq4_reranking.csv', index=False)
sens = []
for order in itertools.permutations(CRITERION_ORDER):
    oc = df.apply(lambda r: rerank(r, list(order)), axis=1)
    sens.append(dict(order=' > '.join(o.replace('_', ' ') for o in order), reversed_pct=100*(oc == 'reversed').mean(), stable_pct=100*(oc == 'stable').mean(), unresolved_pct=100*(oc == 'unresolved').mean()))
sens = pd.DataFrame(sens); sens.round(3).to_csv(RESULTS / 'rq4_sensitivity.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1, 1.3]})
left = np.zeros(2); cols = {'stable': NAVY, 'reversed': GOLD, 'unresolved': TEAL, 'insufficient evidence': GREY}
for o_ in OUTC:
    vals = np.array([t4[(t4.group == s) & (t4.outcome == o_)].share_pct.iloc[0] for s in SUBSETS])
    ax1.barh(SUBSETS, vals, left=left, color=cols[o_], label=o_); left += vals
ax1.set_xlabel('% of pairs'); ax1.set_title('(a) Dataset ranking under frozen criteria'); ax1.legend(frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.25), ncol=2)
ax2.barh(sens.order, sens.reversed_pct, color=GREY); ax2.set_xlabel('% reversed'); ax2.set_title('(b) Sensitivity to criterion ordering'); ax2.tick_params(axis='y', labelsize=8)
fig.suptitle('Figure 4. How often the preference ranking survives explicit ethical criteria (lexicon pass)'); fig.tight_layout(); save(fig, 'fig4_rq4_reranking'); plt.show(); plt.close(fig)
display(t4[t4.group.isin(SUBSETS + ['all'])].round(2)); display(sens.round(2))

### 9b Supplementary — two-rule lexical sensitivity (retained from the initial implementation)

In [ ]:
# Compare a safety-marker rule with a third-party acknowledgment rule.
rows=[]
for sub,g in df.groupby('subset'):
    for rule in ['safety_markers','acknowledgment_markers']:
        def scores(side):
            return (g[side+'_refuse']|g[side+'_redirect']|g[side+'_safety_info']).astype(int) if rule=='safety_markers' else g[side+'_ack_third'].astype(int)
        d=scores('chosen')-scores('rejected')
        for label,mask in [('chosen_higher',d>0),('rejected_higher',d<0),('rule_tie',d==0)]:
            rows.append(dict(subset=sub,rule=rule,outcome=label,n_pairs=len(g),count=int(mask.sum()),share=float(mask.mean())))
t=pd.DataFrame(rows);display(t);t.to_csv(RESULTS/'supp_rq4_rule_sensitivity.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,sub in zip(axes,SUBSETS):
    part=t[t.subset==sub].pivot(index='rule',columns='outcome',values='share')
    part.plot.barh(stacked=True,ax=ax,color=['#17365D','#D79B27','#AAB3BC']);ax.set_title(sub);ax.set_xlabel('Fraction of pairs');ax.legend(fontsize=8)
fig.suptitle('RQ4 lexical rule sensitivity — not validated ethical rankings');fig.tight_layout()
for ext in ['pdf','png']:fig.savefig(RESULTS/('supp_rq4_rule_sensitivity.'+ext),bbox_inches='tight',dpi=180)
plt.show();plt.close(fig)
manifest.append('supp_rq4_rule_sensitivity')

## 10 Human-coding validation hook (steps 7–9 of the methodology)
When the coded sheet `main_500_coded.csv` is attached under `HUMAN_INPUT` (or placed next to the notebook), this cell un-blinds it with the key, computes Cohen's κ between the lexicon and the human codes per feature, κ between coders on the double-coded subset where a second coder's file exists, and the human RQ4 outcome distribution. Until then it reports what it would do.

In [ ]:
def find_any(name):
    for root in [HUMAN_INPUT, INPUT_ROOT, Path.cwd(), WORK_ROOT]:
        hits = list(Path(root).rglob(name)) if Path(root).exists() else []
        if hits: return hits[0]
    return None
coded, key = find_any('main_500_coded.csv'), find_any('main_500_KEY.csv')
if coded and key:
    hc = pd.read_csv(coded).merge(pd.read_csv(key), on='pair_id')
    yes = lambda s: s.astype(str).str.strip().str.lower().isin(['1', 'true', 'y', 'yes'])
    res = []
    for f in ['refuse', 'partial', 'redirect', 'safety_info']:
        human = np.where(hc.A_is == 'chosen', yes(hc[f'A_{f}']), yes(hc[f'B_{f}']))
        lex = hc.merge(df[['pair_id', f'chosen_{f}']], on='pair_id')[f'chosen_{f}'].astype(bool)
        res.append(dict(feature=f, kappa_lexicon_vs_human=cohen_kappa_score(human, lex), n=len(hc)))
    rank = hc['ethical_rank(A/B/tie/insufficient_evidence)'].astype(str).str.strip().str.upper()
    hc['human_rq4'] = np.select([rank == hc.A_is.str.upper().str[0], rank.isin(['A', 'B']), rank == 'TIE'], ['stable', 'reversed', 'unresolved'], 'insufficient evidence')
    pd.DataFrame(res).to_csv(RESULTS / 'validation_kappa_lexicon_vs_human.csv', index=False)
    hc.human_rq4.value_counts().rename_axis('outcome').reset_index(name='count').to_csv(RESULTS / 'validation_human_rq4.csv', index=False)
    second = find_any('main_500_coded_coder2.csv')
    if second:
        c2 = pd.read_csv(second).merge(hc[['pair_id'] + [f'A_{f}' for f in ['refuse','partial','redirect','safety_info']]], on='pair_id', suffixes=('_c2', '_c1'))
        kk = [dict(feature=f, kappa_coder1_vs_coder2=cohen_kappa_score(yes(c2[f'A_{f}_c1']), yes(c2[f'A_{f}_c2'])), n=len(c2)) for f in ['refuse','partial','redirect','safety_info']]
        pd.DataFrame(kk).to_csv(RESULTS / 'validation_kappa_coders.csv', index=False); display(pd.DataFrame(kk))
    display(pd.DataFrame(res)); display(hc.human_rq4.value_counts())
else:
    print('Human-coded sheet not found. After coding, save the blinded sheet as main_500_coded.csv (and the second coder\'s as main_500_coded_coder2.csv),')
    print('attach it with the KEY file, and re-run this cell: it reports kappa per feature and the human RQ4 outcome distribution.')

## 11 Package results
Writes a manifest and zips all research outputs, diagnostics and annotation templates (keys stay in the private folder and are not zipped).

In [ ]:
save_json({'code_version': CODE_VERSION, 'analysis_split': ANALYSIS_SPLIT, 'pairs_analysed': int(len(df)),
           'conflict_pairs': int(df.conflict.sum()), 'third_party_cases': int(df.third_case.sum()),
           'outputs': manifest, 'status': 'automated lexicon pass; human validation runs via step 10',
           'limitations': ['Lexicon matches do not verify factual accuracy or ethical adequacy.',
                           'Results describe the analysed split of one 2022 annotator pool under one instruction set.']}, RESULTS / 'manifest.json')
zip_path = RUN / 'final_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder, arc in [(RESULTS, 'research_outputs'), (DIAG, 'diagnostics'), (TEMPLATES, 'annotation_templates')]:
        for p in sorted(folder.iterdir()): z.write(p, f'{arc}/{p.name}')
print('Results:', zip_path, f'({zip_path.stat().st_size/1e6:.2f} MB)')
for folder in [RESULTS, DIAG, TEMPLATES]:
    print(folder.name + '/'); [print('   ', p.name) for p in sorted(folder.iterdir())]